In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from pathlib import Path
from utils import *
from experiment import Experiment
from ReasoningGraph import ReasoningGraph
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# model_name = "Qwen/Qwen3-4B-Thinking-2507"
# model_name = "Qwen/Qwen3-1.7B"
model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
num_problems = 2
num_rollouts = 3
temperature = 1.0

In [4]:
dataset = 'openai/gsm8k'
problems = load_problems_from_dataset(num_problems=num_problems)

for i, p in enumerate(problems[:3]):
    print(f"\nProblem {i+1}: {p['question'][:100]}...")
    print(f"Ground truth: {p['ground_truth']}")

Loading 2 problems from openai/gsm8k:test

Problem 1: Two apples fell out of the tree, and one of them landed on Newton's head.  Newton picked up the two ...
Ground truth: 11.0

Problem 2: Nancy is returning her overdue books to the library. She owes $0.50 cents each on 8 books, plus a fl...
Ground truth: 6.0

Problem 1: Two apples fell out of the tree, and one of them landed on Newton's head.  Newton picked up the two ...
Ground truth: 11.0

Problem 2: Nancy is returning her overdue books to the library. She owes $0.50 cents each on 8 books, plus a fl...
Ground truth: 6.0


In [5]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

In [8]:
# Create an experiment
experiment = Experiment(model_name, problems, num_rollouts, temperature)

# Setup the experiment
experiment.setup()

Created experiment at: experiments/experiment_20251027_155017 with config:
{'model_name': 'Qwen/Qwen2.5-Math-1.5B-Instruct', 'num_problems': 2, 'num_rollouts': 3, 'temperature': 1.0, 'timestamp': '20251027_155017'}


In [9]:
# Conduct the experiment with initiailized model and tokenizer
experiment.conduct_experiment(model, tokenizer)


Problem 1/2



Problem 2/2



Experiment completed! Results saved to: experiments/experiment_20251027_155017


In [24]:
tokens = [tokenizer.decode(id) for id in output_ids[len(model_inputs.input_ids[0]):]]
entropies = reasoning_graph.metrics
visualize_tokens(tokens, entropies)

In [25]:
# Example of loading a saved reasoning graph
loaded_graph = ReasoningGraph.load(output_dir)

# Verify the loaded data
print("Metrics length:", len(loaded_graph.metrics))
print("Probability distributions shape:", loaded_graph.prob_distributions[0].shape)
print("Node cutoff value:", loaded_graph.node_cutoff)

# You can now use this loaded graph for visualization or analysis

Metrics length: 318
Probability distributions shape: torch.Size([151936])
Node cutoff value: 0.058349609375
